Assignment 3
Compare BERT to interpretable classifiers for text classification

1. Download the ReviewBase dataset
Split dotat on space character
The data set also contains a class labael associated with each text

In [5]:
file_training = open('ReviewBaseTraining.txt', 'r', encoding = 'utf-8')
training_data_string = file_training.read()
file_training.close()

file_validation = open('ReviewBaseValidation.txt', 'r', encoding = 'utf-8')
validation_data_string = file_validation.read()
file_validation.close()

file_test = open('ReviewBaseTest.txt', 'r', encoding = 'utf-8')
test_data_string = file_test.read()
file_test.close()


In [6]:
training_data = training_data_string.splitlines()
validation_data = validation_data_string.splitlines()
test_data = test_data_string.splitlines()




def split_labels(data):
    current_label = None
    x = []
    y = []
    for word in data:
        label, text = word.split("\t")
        x.append(text)
        y.append(label)
    return x, y

x_train, y_train = split_labels(training_data)
x_val, y_val = split_labels(validation_data)
x_test, y_test = split_labels(test_data)

y_train = [int(label) for label in y_train]
y_val = [int(label) for label in y_val]
y_test = [int(label) for label in y_test]

print(x_train[:2])
print(y_train[:2])



["Story of a man who has unnatural feelings for a pig . Starts out with a opening scene that is a terrific example of absurd comedy . A formal orchestra audience is turned into an insane , violent mob by the crazy chantings of it ' s singers . Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting . Even those from the era should be turned off . The cryptic dialogue would make Shakespeare seem easy to a third grader . On a technical level it ' s better than you might think with some good cinematography by future great Vilmos Zsigmond . Future stars Sally Kirkland and Frederic Forrest can be seen briefly .", 'Bromwell High is a cartoon comedy . It ran at the same time as some other programs about school life , such as " Teachers " . My 35 years in the teaching profession lead me to believe that Bromwell High \' s satire is much closer to reality than is " Teachers " . The scramble to survive financially , the insightful students 

Set up and fine-tune the DeBERTa-v3 model using low-rank adaptiation(LoRA)
Use a single fully connected layer (after transformer bloacks)


In [ ]:
!pip install -q transformers[sentencepiece] accelerate

!pip install peft

In [7]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from peft import LoraConfig, get_peft_model

from torch.utils.data import Dataset, DataLoader

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

In [8]:
#Fine tuning involves setting suitable values for the weights int he feedforward layer
#Slightly modifying weights in BERT itself
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

train_data = [(x, y) for x, y in zip(x_train, y_train)]
test_data = [(x, y) for x, y in zip(x_test, y_test)]
validation_data = [(x, y) for x, y in zip(x_val, y_val)]

dataloaded_train = DataLoader(train_data, batch_size = 32, shuffle = True)


data_loader_test = DataLoader(test_data, batch_size = 32, shuffle = False)
data_loader_val = DataLoader(validation_data, batch_size = 32, shuffle = False)




lora_config = LoraConfig(r = 8, target_modules = ["query_proj", "key_proj", "value_proj"])

model = get_peft_model(model, lora_config)
model.to(device)

trainable = 0
total = 0
for p in model.parameters():
  total += p.numel()
  if p.requires_grad:
    trainable += p.numel()
print(f"Trainable params: {trainable:,}")
print(f"Total params {total:,}")
print(f"Trainable: {100*trainable/total:.4f}%")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

Trainable params: 221,184
Total params 142,117,634
Trainable: 0.1556%


In [5]:

import torch
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

cuda available: True
device count: 1
gpu name: Tesla T4


In [ ]:

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-4)

epochs = 2

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    steps = 0
    for batch in dataloaded_train:
        texts, labels = batch

        train_x_token = tokenizer(list(texts), return_tensors="pt", truncation = True, padding = True, max_length = 128).to(device)

        labels_t = labels.to(device)
        #optimizer = "hddell"
        #Actual training
        out = model(input_ids = train_x_token["input_ids"], labels = labels_t, attention_mask = train_x_token["attention_mask"])
        loss = out.loss


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        steps += 1



    print(f"epoch {epoch+1}, avg_loss = {running_loss/steps}")


Set up and train a linear perceptron using n-gram for the same task

Run for n_max = 1 (unigrams onlt), n_max =2 (unigrams and bigrams) and n_max = 3 (unigrams, bigrams and trigrams).

for each n, try different minimum instance counts (cmin) for feature inclusion (1,2,10,100)
Then extract the model with highest validation accuracy (for each n) and compute its accuracy over the test set

In [ ]:
from sklearn.datasets import make_classification


from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt

import random

In [ ]:
thresh = 0
n_features = 2
c_min = 3

def classify(weights, x, bias, threshold):
  y = x.dot(weights).item() + bias
  if y>=threshold:
    return 1
  else:
    return 0

def accuracy_calc(weights, vectorized_val, bias, labels, threshold = thresh):
  correct = 0
  total = vectorized_val.shape[0]
  for i in range(total):
    vec_val = vectorized_val[i]
    pred = classify(weights, vec_val, bias, threshold)
    if pred == labels[i]:
      correct += 1
  accuracy = correct / total
  return accuracy



def linear_perceptron(n, texts, c_min, labels, eta, epochs, threshold = thresh):
  vectorizer = CountVectorizer(ngram_range=(1, n), min_df = c_min)
  vectorized = vectorizer.fit_transform(texts)
  weights = np.zeros(vectorized.shape[1])
  bias = 0.0

  for epoch in range(epochs):
    if epoch % 10 == 0:
      print(epoch)

    order = np.random.permutation(len(texts))

    for i in order:
      #x_i = vectorized[i].toarray().ravel().astype(float)
      #x_dense = x_i.toarray().ravel()
      predicted = classify(weights, vectorized[i], bias, threshold)
      if predicted != labels[i]:
        row = vectorized[i]
        indices = row.indices
        values = row.data
        weights[indices] += eta * (labels[i] - predicted) * values
        bias += eta * (labels[i] - predicted)

  return weights, vectorizer, bias

In [8]:



#Prediction: Perceptron calculates the weighted total
#of input features and weights in order to provide a
#forecase for particular input

#Activation: Outputs 1 if sumf(x) is greater or equal to a threshold

#Updating weight: If misclassification was made

eta = 0.01
epochs = 50
labels = y_train

c_mins = [1, 3, 10, 100]

y_val = [int(l) for l in y_val]

weights_list = []
biases_list = []
for n in range(1, 4):
  best_accuracy = -1
  best_c_min = None
  best_weights = None
  best_vectorizer = None
  best_bias = None
  local_weights = []
  local_biases = []


  for c_min in c_mins:
    weights, vectorizer, bias = linear_perceptron(n, x_train, c_min, y_train, eta, epochs)
    vectorized_val = vectorizer.transform(x_val)

    accuracy = accuracy_calc(weights, vectorized_val, bias, y_val)
    if accuracy >= best_accuracy:
      best_accuracy = accuracy
      best_c_min = c_min
      best_weights = weights
      best_vectorizer = vectorizer
      best_vectorizer_val = vectorized_val
      best_bias = bias
    local_weights.append(weights)
    local_biases.append(bias)

  weights_list.append(local_weights)
  biases_list.append(local_biases)

  print(f"for n = {n}, the best c_min is {best_c_min} with accuracy {best_accuracy}")

  vectorised_test_val = best_vectorizer.transform(x_test)
  test_accuracy = accuracy_calc(best_weights, vectorised_test_val, best_bias, y_test)

#Repeat: Each input point goes throguh point 2-4


0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 1, the best c_min is 1 with accuracy 0.8304
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 2, the best c_min is 1 with accuracy 0.8426
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 3, the best c_min is 10 with accuracy 0.8352


Faster version of previous !?

In [11]:
thresh = 0
n_features = 2
c_min = 3

def classify(weights, x, bias, threshold):
  y = x.dot(weights).item() + bias
  if y>=threshold:
    return 1
  else:
    return 0

def accuracy_calc(weights, vectorized_val, bias, labels, threshold = thresh):
  scores = vectorized_val @ weights + bias
  preds = (scores >= threshold).astype(int)
  labels = np.asarray(labels, dtype=int)
  accuracy = (preds == labels).mean()
  return accuracy



def linear_perceptron(n, texts, c_min, labels, eta, epochs, threshold = thresh):
  vectorizer = CountVectorizer(ngram_range=(1, n), min_df = c_min)
  vectorized = vectorizer.fit_transform(texts)
  weights = np.zeros(vectorized.shape[1])
  bias = 0.0

  for epoch in range(epochs):
    if epoch % 10 == 0:
      print(epoch)

    order = np.random.permutation(len(texts))

    for i in order:
      #x_i = vectorized[i].toarray().ravel().astype(float)
      #x_dense = x_i.toarray().ravel()
      score = vectorized[i].dot(weights) + bias
      if score >= threshold:
        predicted = 1
      else:
        predicted = 0
      if predicted != labels[i]:
        row = vectorized[i]
        indices = row.indices
        values = row.data
        dif_eta = eta*(labels[i] - predicted)
        weights[indices] +=dif_eta * values
        bias += dif_eta

  return weights, vectorizer, bias


#Prediction: Perceptron calculates the weighted total
#of input features and weights in order to provide a
#forecase for particular input

#Activation: Outputs 1 if sumf(x) is greater or equal to a threshold

#Updating weight: If misclassification was made

eta = 0.01
epochs = 50
labels = y_train

c_mins = [1, 3, 10, 100]

y_val = [int(l) for l in y_val]
weights_list = []
biases_list = []
for n in range(1, 4):
  best_accuracy = -1
  best_c_min = None
  best_weights = None
  best_vectorizer = None
  best_bias = None
  local_weights = []
  local_biases = []

  for c_min in c_mins:
    weights, vectorizer, bias = linear_perceptron(n, x_train, c_min, y_train, eta, epochs)
    vectorized_val = vectorizer.transform(x_val)

    accuracy = accuracy_calc(weights, vectorized_val, bias, y_val)
    if accuracy >= best_accuracy:
      best_accuracy = accuracy
      best_c_min = c_min
      best_weights = weights
      best_vectorizer = vectorizer
      best_vectorizer_val = vectorized_val
      best_bias = bias
    local_weights.append(weights)
    local_biases.append(bias)

  weights_list.append(local_weights)
  biases_list.append(local_biases)

  print(f"for n = {n}, the best c_min is {best_c_min} with accuracy {best_accuracy}")

  vectorised_test_val = best_vectorizer.transform(x_test)
  test_accuracy = accuracy_calc(best_weights, vectorised_test_val, best_bias, y_test)

#Repeat: Each input point goes throguh point 2-4

0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 1, the best c_min is 1 with accuracy 0.8086680761099366
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 2, the best c_min is 10 with accuracy 0.8155391120507399
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 3, the best c_min is 3 with accuracy 0.8205602536997886


Plot (diagram) showing all the unigrams, bigrams and trigrams, with a green bar for positive weights and red bar for negative weights.
Length of bar propotional to weight magnitude
Apply this visualizer to a few shorter reviews

In [ ]:
import matplotlib.pyplot as plt
import random

In [ ]:
def plot( weights, vect, words):
  n_grams = words.copy()
  for i in range(len(words) -1):
    n_grams.append(words[i] + " " + words[i+1])
  for i in range(len(words) -2):
    n_grams.append(words[i] + " " + words[i+1] + " " + words[i+2])
  words_in_voc = vect.vocabulary_
  indexes = []
  plot_words = []
  colours = []
  weights_used = []

  for gram in n_grams:
    if gram in words_in_voc:
      idx = words_in_voc[gram]
      weight = weights[idx]
    else:
      weight = 0.0

    plot_words.append(gram)
    weights_used.append(weight)
    if weight > 0:
      colours.append("green")
    elif weight == 0:
      colours.append("gray")
    else:
      colours.append("red")

  plt.figure(figsize=[10,10])
  plt.bar(plot_words, weights_used, color = colours)
  plt.title("Commonality of different n grams in sentence")
  plt.xlabel("n_grams")
  plt.ylabel("Weight")
  plt.xticks(rotation=90)
  plt.tight_layout()

  plt.show()

num_graphs = 3

for _ in range(num_graphs):
  short_sentence = False
  while not short_sentence:
    rnd = random.randint(0, len(x_test)-1)
    words = x_test[rnd].split()
    if 15 <= len(words)<= 25:
      short_sentence = True
      sentence_num = rnd
      sentence = x_test[sentence_num]

  plot(best_weights, best_vectorizer, words)






Review Extended consists of a training and a validation set.
Train linear perceptron model with n_max = 1, n_max = 2, n_max = 3
Find higherst accuracy
For each n, average the weights from the orginal base model with the weights obtained for the extended model as w_c = alpha * w_b + (1 - alpha) * w_e
w_b: weight from base
w_e: weight from extended
(if w_b does not exist, w_c = w_e)
Chose the alpha for each n that maximises accuracy over validation set from base

In [ ]:
file_training_extended = open('ReviewExtendedTraining.txt', 'r', encoding = 'utf-8')
training_data_string_extended = file_training_extended.read()
file_training_extended.close()



file_validation_extended = open('ReviewExtendedValidation.txt', 'r', encoding = 'utf-8')
validation_data_string_extended = file_validation_extended.read()
file_validation_extended.close()

training_data_extended = training_data_string_extended.splitlines()
validation_data_extended = validation_data_string_extended.splitlines()

def split_labels_ext(data):
    current_label = None
    x = []
    y = []
    for i, word in enumerate(data):
      if "\t" in word:
        label, text = word.split("\t", 1)
      else:
        label, text = word.split(None, 1)
      if label== "0" or label == "1":
        x.append(text)
        y.append(int(label))
    return x, y

#def split_labels(data):
#    current_label = None
#    x = []
#    y = []
#    i = 0
#    for word in data:

 #     label, text = word[0], word[1:]
 ##     x.append(text)
  #    if label != "0" and label != "1":
  #      print("error", word)
  #    y.append(int(label))
  #    i+=1
  #  return x, y


x_train_ex, y_train_extended = split_labels_ext(training_data_extended)
x_val_ex, y_val_extended = split_labels_ext(validation_data_extended)

y_train_ex = [int(label) for label in y_train_extended]
y_val_ex = [int(label) for label in y_val_extended]


In [ ]:
c_mins = [1, 3, 10, 100]
ex_weights = []
ex_biases = []
chosen_c_mins = []

In [ ]:
n = 1
best_accuracy_ex = -1
best_c_min_ex = None
best_weights_ex = None
best_vectorizer_ex = None
best_bias_ex = None
best_i = None

for i, c_min in enumerate(c_mins):
  weights, vectorizer, bias = linear_perceptron(n, x_train_ex, c_min, y_train_ex, eta, epochs)
  vectorized_val = vectorizer.transform(x_val_ex)
  accuracy = accuracy_calc(weights, vectorized_val, bias, y_val_ex)

  if accuracy >= best_accuracy_ex:
    best_accuracy_ex = accuracy
    best_c_min_ex = c_min
    best_weights_ex = weights
    best_vectorizer_ex = vectorizer
    best_vectorizer_val_ex = vectorized_val
    best_bias_ex = bias
    best_i = i

ex_weights.append(best_weights_ex)
ex_biases.append(best_bias_ex)
chosen_c_mins.append(best_i)


print(f"for n = {n}, the best c_min is {best_c_min_ex} with accuracy {best_accuracy_ex}")


In [3]:
n = 2
best_accuracy_ex = -1
best_c_min_ex = None
best_weights_ex = None
best_vectorizer_ex = None
best_bias_ex = None
best_i = None

for i, c_min in enumerate(c_mins):
  weights, vectorizer, bias = linear_perceptron(n, x_train_ex, c_min, y_train_ex, eta, epochs)
  vectorized_val = vectorizer.transform(x_val_ex)
  accuracy = accuracy_calc(weights, vectorized_val, bias, y_val_ex)

  if accuracy >= best_accuracy_ex:
    best_accuracy_ex = accuracy
    best_c_min_ex = c_min
    best_weights_ex = weights
    best_vectorizer_ex = vectorizer
    best_vectorizer_val_ex = vectorized_val
    best_bias_ex = bias
    best_i = i

ex_weights.append(best_weights_ex)
ex_biases.append(best_bias_ex)
chosen_c_mins.append(best_i)


print(f"for n = {n}, the best c_min is {best_c_min_ex} with accuracy {best_accuracy_ex}")


NameError: name 'linear_perceptron' is not defined

In [ ]:
n = 3
best_accuracy_ex = -1
best_c_min_ex = None
best_weights_ex = None
best_vectorizer_ex = None
best_bias_ex = None
best_i = None

for i, c_min in enumerate(c_mins):
  weights, vectorizer, bias = linear_perceptron(n, x_train_ex, c_min, y_train_ex, eta, epochs)
  vectorized_val = vectorizer.transform(x_val_ex)
  accuracy = accuracy_calc(weights, vectorized_val, bias, y_val_ex)

  if accuracy >= best_accuracy_ex:
    best_accuracy_ex = accuracy
    best_c_min_ex = c_min
    best_weights_ex = weights
    best_vectorizer_ex = vectorizer
    best_vectorizer_val_ex = vectorized_val
    best_bias_ex = bias
    best_i = i

ex_weights.append(best_weights_ex)
ex_biases.append(best_bias_ex)
chosen_c_mins.append(best_i)


print(f"for n = {n}, the best c_min is {best_c_min_ex} with accuracy {best_accuracy_ex}")


In [ ]:
#Delete this and run the previous ones
for n in range(1, 4):
  best_accuracy_ex = -1
  best_c_min_ex = None
  best_weights_ex = None
  best_vectorizer_ex = None
  best_bias_ex = None
  best_i = None

  for i, c_min in enumerate(c_mins):
    weights, vectorizer, bias = linear_perceptron(n, x_train_ex, c_min, y_train_ex, eta, epochs)
    vectorized_val = vectorizer.transform(x_val_ex)
    accuracy = accuracy_calc(weights, vectorized_val, bias, y_val_ex)

    if accuracy >= best_accuracy_ex:
      best_accuracy_ex = accuracy
      best_c_min_ex = c_min
      best_weights_ex = weights
      best_vectorizer_ex = vectorizer
      best_vectorizer_val_ex = vectorized_val
      best_bias_ex = bias
      best_i = i

  ex_weights.append(best_weights_ex)
  ex_biases.append(best_bias_ex)
  chosen_c_mins.append(best_i)


  print(f"for n = {n}, the best c_min is {best_c_min_ex} with accuracy {best_accuracy_ex}")
##################################

0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
for n = 1, the best c_min is 1 with accuracy 0.84306
0
10
20
30
40
0
10
20
30
40
0
10
20
30
40
0
10


FAster version of previous?!

For each n, average the weights from the orginal base model with the weights obtained for the extended model as w_c = alpha * w_b + (1 - alpha) * w_e w_b: weight from base w_e: weight from extended (if w_b does not exist, w_c = w_e) Chose the alpha for each n that maximises accuracy over validation set from base



In [ ]:

alphas = [0.01, 0.05, 0.1, 0.5]

n_to_w_ex = {best_vectorizer_ex.get_feature_names_out()[i]: ex_weights[i] for i in range(len(ex_weights))}
n_to_w = {best_vectorizer.get_feature_names_out()[i]: weights[i] for i in range(len(weights))}

exs_words = best_vectorizer_ex.get_feature_names_out()
words = best_vectorizer.get_feature_names_out()

all_words = list(set(exs_words + words))

for alpha in alphas:
  weights_final = []
  for word in all_words:
    w_e = n_to_w_ex.get(word, None)
    w_b = n_to_w.get(word, None)

    if w_e == None:
      w_c = w_b
    elif w_b == None:
      w_c = w_e
    else:
      w_c = alpha*w_b + (1-alpha)* w_e

    weights_final.append(w_c)

  b_c = alpha * local_biases + (1-alpha) * ex_biases
  accuracy = accuracy_calc(weights_final, vectorized_val, x_val, b_c, y_val)
  print(f"For alpha {alpha}, accuracy is {accuracy}")





